In [0]:
%run ./00_config

In [0]:
%sql
-- Garante que conseguimos ler a Bronze antes de fazer o ETL
CREATE SCHEMA IF NOT EXISTS hive_metastore.crypto_bronze
LOCATION 'abfss://bronze@stcryptolakehouse.dfs.core.windows.net/';

CREATE EXTERNAL TABLE IF NOT EXISTS hive_metastore.crypto_bronze.crypto_prices_raw
USING DELTA
LOCATION 'abfss://bronze@stcryptolakehouse.dfs.core.windows.net/crypto_prices_raw';

In [0]:
from pyspark.sql.functions import col, from_unixtime, to_timestamp, current_timestamp

df_bronze = spark.read.table("hive_metastore.crypto_bronze.crypto_prices_raw")

df_silver = df_bronze \
    .withColumn("data_hora_cotacao", to_timestamp(from_unixtime(col("timestamp_api")))) \
    .withColumn("timestamp_processamento_silver", current_timestamp()) \
    .dropDuplicates(["moeda_id", "timestamp_api"])

caminho_silver = "abfss://silver@stcryptolakehouse.dfs.core.windows.net/crypto_prices_clean"
df_silver.write.format("delta").mode("overwrite").save(caminho_silver)

In [0]:
%sql
-- Apresenta o resultado limpo para o catálogo
CREATE SCHEMA IF NOT EXISTS hive_metastore.crypto_silver
LOCATION 'abfss://silver@stcryptolakehouse.dfs.core.windows.net/';

CREATE EXTERNAL TABLE IF NOT EXISTS hive_metastore.crypto_silver.crypto_prices_clean
USING DELTA
LOCATION 'abfss://silver@stcryptolakehouse.dfs.core.windows.net/crypto_prices_clean';